# 3 Preprocessing and Mapping of Passive and EMA Data for Paper "Towards JITAI"

This notebook shows the analysis of situational context using EMA and passive sensing data

1. **Load Data**: Load necessary data from pickle files.
2. **Preprocess EMA**:

In [53]:
from pathlib import Path

import sys

%load_ext autoreload
%autoreload 2
import logging
# Configure logging
logging.basicConfig(level=logging.INFO)

# --- Paths / imports -------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
PREPROCESSING_DIR = PROJECT_ROOT / "functions" / "preprocessing"
for p in (PROJECT_ROOT, PREPROCESSING_DIR):
    if str(p) not in sys.path:
        sys.path.append(str(p))

import pickle
from server_config import redcap_path, preprocessed_path_jitai

import pandas as pd
import numpy as np

from functions.preprocessing import gps_features
from functions.preprocessing.ema_mappings import run_ema_mappings
from functions.preprocessing.missing_data import summarize_missing_data


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [54]:
import warnings
# Suppress only SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

In [55]:
backup_path = preprocessed_path_jitai+ "/backup_passive_recent.feather"
df_backup = pd.read_feather(backup_path)

with open(preprocessed_path_jitai + '/ema_content.pkl', 'rb') as file:
    df_ema_content = pickle.load(file)  

with open(preprocessed_path_jitai + '/monitoring_data.pkl', 'rb') as file:
    df_monitoring = pickle.load(file)

with open(redcap_path + '/redcap_data_jitai_28102025.pkl', 'rb') as file:
    df_redcap = pickle.load(file)

split_path = redcap_path + "/sp1_cleaned"+"/111025_subject_splits.csv"
df_split = pd.read_csv(split_path)


In [56]:
cust_list = df_ema_content.customer.unique().tolist()
df_redcap_original = df_redcap.dropna(subset = ["age", "customer"] )
df_redcap_original = df_redcap_original[df_redcap_original.customer.isin(cust_list)]
df_redcap_original = df_redcap_original.drop_duplicates(subset="customer") 
cust_list_final = df_redcap_original.customer.unique().tolist()

df_backup = df_backup.loc[df_backup.customer.isin(cust_list_final)]


In [57]:
# Configurations
# Check min. amount of EMA data available to map to passive data

#GPS data
speed_limit = 1.4
max_distance = 150 
kms_per_radian = 6371000
epsilon = 100/kms_per_radian
min_samples = 10
min_cluster_size = 20
min_nights_obs = 4
min_f_home = 0.5

# EMA
assessment_phase = [0] #1,2
min_num_daily = 4
min_days_data = 7


#Passive to EMA matching
timedelta_hours = 2
assess = 0

## Filter for participants with sufficient data

In [58]:
# first assessment phase finished
df_ema = df_ema_content.loc[df_ema_content.status.isin(["Abgeschlossen", "Post_Erhebung_1",
                                                             "Erhebung_2_aktiv","Post_Erhebung_2", "Erhebung_3_aktiv", "Dropout"])]


In [59]:
df_ema_content = df_ema_content.loc[df_ema_content.customer.isin(cust_list_final)]

In [60]:
df_ema_content["local_start_time"] = (
    df_ema_content["quest_create"] + df_ema_content["inferred_tzoffset_timedelta"]
).dt.tz_localize(None)

In [61]:
#off_ema = pd.to_timedelta(df_ema["inferred_tzoffset"].fillna(0), unit="m")

#df_ema.quest_create = (
#    pd.to_datetime(df_ema.quest_create, utc=True, errors="coerce") + off_ema).dt.tz_localize(None)


In [62]:
df_ema["quest_create_day"] = df_ema.quest_create.dt.normalize()
df_ema["quest_create_hour"] = df_ema.quest_create.dt.hour

In [63]:
extra_cols = ["assess", "study", "quest_create", "weekend", "quest_nr", "weekday", "season", "time_of_day", "quest_create_day", "quest_create_hour"]

aggregated_info = df_ema.groupby(["customer", "unique_day_id"])[extra_cols].first().reset_index()


In [64]:
df_ema_panas = df_ema.loc[df_ema.quest_title.isin(['panas_fear1', 'panas_fear2', 'panas_guilt1', 
            'panas_guilt2', 'panas_hostility1', 'panas_hostility2', 
             'panas_sadness1', 'panas_sadness2'])]

# Pivot the table as specified
df_piv = df_ema_panas.pivot_table(
    index=["customer", "unique_day_id", "assess"],
    columns="quest_title",
    values="choice_text",
    aggfunc='first'  # Using 'first' since each entry should theoretically be unique per group
)

# The columns are now a single level Index with just the quest_title values since 'values' is not a list anymore
df_piv.columns = [col for col in df_piv.columns.values]

# Reset the index to turn the MultiIndex into columns
df_piv = df_piv.reset_index()
df_piv = df_piv.drop_duplicates()


In [65]:
df_piv = df_piv.loc[df_piv.assess==0] # first assessment phase


In [66]:
df_piv.shape

(26349, 11)

### Add situation features 

In [67]:
df_sit = df_ema[df_ema.quest_title =="situation1"]

In [68]:
# Pivot the table as specified
df_sit = df_sit.pivot_table(
    index=["customer", "unique_day_id", "quest_create", "choice_id"],
    columns="quest_title",
    values="choice_text",
    aggfunc='first'  # Using 'first' since each entry should theoretically be unique per group
)

# The columns are now a single level Index with just the quest_title values since 'values' is not a list anymore
df_sit.columns = [col for col in df_sit.columns.values]

# Reset the index to turn the MultiIndex into columns
df_sit = df_sit.reset_index()

df_sit = df_sit.drop_duplicates(subset=['customer', 'unique_day_id', 'choice_id'])

# Define the situation mapping
situation_mapping = {
    'Etwas Anderes': 'other',  
    'Kümmern um Kinder / Angehörige': 'care work',
    'Unterwegs (z.B. in der U-Bahn)': 'travelling',
    'Hausarbeit oder Erledigungen': 'chores',
    'Essen/ Trinken/ Körperpflege': 'eating - drinking - selfcare',  # Changed commas to hyphens
    'Freizeitaktivität, eher aktiv (z.B. Sport, Unternehmungen)': 'active leisure',
    'Smartphone/ Soziale Medien': 'smartphone - social media',  # Changed commas to hyphens
    'Freizeitaktivität, eher passiv (z.B. Film schauen, Lesen)': 'passive leisure',
    'Arbeit oder Studium': 'work or study'
}


In [69]:
df_sit['situation1_simplified'] = df_sit['situation1'].map(situation_mapping)

In [70]:
df_sit_day = (
    df_sit[["customer", "unique_day_id", "situation1_simplified"]]
    .drop_duplicates(subset=["customer", "unique_day_id", "situation1_simplified"])
    .assign(value=1)
    .pivot_table(
        index=["customer", "unique_day_id"],
        columns="situation1_simplified",
        values="value",
        aggfunc="max",
        fill_value=0
    )
    .reset_index()
)

df_sit_day.columns = [
    col if col in ["customer", "unique_day_id"] else f"situation1_{col}"
    for col in df_sit_day.columns
]

df_sit_day.head()

,customer,unique_day_id,situation1_active leisure,situation1_care work,situation1_chores,situation1_eating - drinking - selfcare,situation1_other,situation1_passive leisure,situation1_smartphone - social media,situation1_travelling,situation1_work or study
0,05kz,20231010_2.0,0,0,0,0,1,0,0,0,0
1,05kz,20231010_3.0,0,1,0,0,0,0,0,1,0
2,05kz,20231010_5.0,0,1,0,0,0,0,0,1,0
3,05kz,20231010_6.0,0,0,1,1,0,1,1,0,0
4,05kz,20231010_7.0,0,0,0,1,0,1,0,0,0


In [71]:
df_piv = df_piv.merge(df_sit_day, on=["customer", "unique_day_id"], how="left")

### Aggregate NA Scales 

In [72]:

na_scale = ['panas_fear1', 'panas_fear2', 'panas_guilt1', 
            'panas_guilt2', 'panas_hostility1', 'panas_hostility2', 
             'panas_sadness1', 'panas_sadness2']

# Step 1: Ensure the columns in pa_scale and na_scale are numeric
df_piv[na_scale] = df_piv[na_scale].apply(pd.to_numeric, errors='coerce')
# Drop rows where any of the na_scale columns have NaN
df_piv = df_piv.dropna(subset=na_scale, how='any')

# Step 2: Calculate the mean for PA and NA scales per unique_day_id
df_piv['mean_na'] = df_piv.groupby(['customer', 'unique_day_id'])[na_scale].transform('mean').mean(axis=1)
df_piv['mean_fear'] = df_piv.groupby(['customer', 'unique_day_id'])[['panas_fear1', 'panas_fear2']].transform('mean').mean(axis=1)
df_piv['mean_guilt'] = df_piv.groupby(['customer', 'unique_day_id'])[['panas_guilt1', 'panas_guilt2']].transform('mean').mean(axis=1)
df_piv['mean_hostility'] = df_piv.groupby(['customer', 'unique_day_id'])[['panas_hostility1', 'panas_hostility2']].transform('mean').mean(axis=1)
df_piv['mean_sadness'] = df_piv.groupby(['customer', 'unique_day_id'])[['panas_sadness1', 'panas_sadness2']].transform('mean').mean(axis=1)


In [73]:
df_piv = pd.merge(aggregated_info, df_piv, on=["customer","assess", "unique_day_id"])


In [74]:
# Calculate the number of unique 'unique_day_id' per 'customer' and 'quest_complete_day'
df_piv['n_quest'] = df_piv.groupby(['customer', 'quest_create_day'])['unique_day_id'].transform('nunique')

# 1) For each customer, count how many UNIQUE days they have n_quest >= min_num_daily
df_days_count = (df_piv[df_piv['n_quest'] >= min_num_daily]
                 .groupby('customer')['quest_create_day'].nunique()
                 .reset_index(name='n_days_min'))

# 2) Identify valid customers
valid_customers = df_days_count.loc[df_days_count['n_days_min'] >= min_days_data, 'customer']

# 3) Filter the original df_piv to keep all rows from valid customers
df_piv_filtered = df_piv[df_piv['customer'].isin(valid_customers)]


In [75]:
df_piv_filtered["n_quest_sum"] = (
    df_piv.groupby('customer')['unique_day_id'].transform('nunique')
)

In [76]:
df_piv_filtered.customer.nunique()

280

## 1. Prepare passive features

In [77]:
df_pass_act = df_backup.copy()

In [78]:
# Only keep data that were collected during the first assessment phase
df_pass_act_base = df_pass_act[df_pass_act.startTimestamp <= (df_pass_act.ema_base_start + pd.Timedelta(days=16))]

In [79]:
df_pass_act_base = df_pass_act_base.loc[df_pass_act_base.customer.isin(valid_customers)]

In [80]:
#off = pd.to_timedelta(df_pass_act_base["inferred_tzoffset"].fillna(0), unit="m")

#df_pass_act_base.startTimestamp = (pd.to_datetime(df_pass_act_base.startTimestamp, utc=True, errors="coerce") + off).dt.tz_localize(None)
#df_pass_act_base.endTimestamp = (pd.to_datetime(df_pass_act_base.endTimestamp, utc=True, errors="coerce") + off).dt.tz_localize(None)

### 1.1 Calculate GPS features

In [81]:
df_pass_act_base

,customer,for_id,type,startTimestamp,endTimestamp,start_end,doubleValue,longValue,booleanValue,startTimestamp_day,startTimestamp_hour,ema_base_start,study_version,createdAt,inferred_tzoffset,inferred_tzoffset_source,local_start_time,local_end_time
0,4MLe,FOR11905,Steps,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,60.0,6.000,NaN,<NA>,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine,2023-05-17 15:44:00,2023-05-17 15:45:00
1,4MLe,FOR11905,ActiveBurnedCalories,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,60.0,0.140,NaN,<NA>,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine,2023-05-17 15:44:00,2023-05-17 15:45:00
2,4MLe,FOR11905,CoveredDistance,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,60.0,4.620,NaN,<NA>,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine,2023-05-17 15:44:00,2023-05-17 15:45:00
3,4MLe,FOR11905,HeartRate,2023-05-17 14:58:01+00:00,2023-05-17 14:58:38+00:00,37.0,NaN,74.0,<NA>,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine,2023-05-17 15:58:01,2023-05-17 15:58:38
4,4MLe,FOR11905,HeartRate,2023-05-17 18:00:55+00:00,NaT,NaN,NaN,95.0,<NA>,2023-05-17 00:00:00+00:00,18.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine,2023-05-17 19:00:55,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127777465,GE6w,FOR12092,CoveredDistance,2025-08-13 18:58:52+00:00,2025-08-13 18:58:55+00:00,3.0,29.717,NaN,<NA>,2025-08-13 00:00:00+00:00,18.0,2025-07-29 00:00:00+00:00,Lang,2025-08-16 07:44:11.112000+00:00,120.0,gps_single,2025-08-13 20:58:52,2025-08-13 20:58:55
127777466,GE6w,FOR12092,Steps,2025-08-13 18:58:52+00:00,2025-08-13 18:58:55+00:00,3.0,16.000,NaN,<NA>,2025-08-13 00:00:00+00:00,18.0,2025-07-29 00:00:00+00:00,Lang,2025-08-16 07:44:11.112000+00:00,120.0,gps_single,2025-08-13 20:58:52,2025-08-13 20:58:55
127777467,GE6w,FOR12092,ActiveBurnedCalories,2025-08-13 20:41:16+00:00,2025-08-13 20:41:28+00:00,12.0,0.329,NaN,<NA>,2025-08-13 00:00:00+00:00,20.0,2025-07-29 00:00:00+00:00,Lang,2025-08-16 07:44:11.113000+00:00,120.0,gps_single,2025-08-13 22:41:16,2025-08-13 22:41:28
127777468,GE6w,FOR12092,CoveredDistance,2025-08-13 20:41:16+00:00,2025-08-13 20:41:28+00:00,12.0,11.645,NaN,<NA>,2025-08-13 00:00:00+00:00,20.0,2025-07-29 00:00:00+00:00,Lang,2025-08-16 07:44:11.114000+00:00,120.0,gps_single,2025-08-13 22:41:16,2025-08-13 22:41:28


In [82]:
df_pass_act_base

,customer,for_id,type,startTimestamp,endTimestamp,start_end,doubleValue,longValue,booleanValue,startTimestamp_day,startTimestamp_hour,ema_base_start,study_version,createdAt,inferred_tzoffset,inferred_tzoffset_source,local_start_time,local_end_time
0,4MLe,FOR11905,Steps,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,60.0,6.000,NaN,<NA>,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine,2023-05-17 15:44:00,2023-05-17 15:45:00
1,4MLe,FOR11905,ActiveBurnedCalories,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,60.0,0.140,NaN,<NA>,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine,2023-05-17 15:44:00,2023-05-17 15:45:00
2,4MLe,FOR11905,CoveredDistance,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,60.0,4.620,NaN,<NA>,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine,2023-05-17 15:44:00,2023-05-17 15:45:00
3,4MLe,FOR11905,HeartRate,2023-05-17 14:58:01+00:00,2023-05-17 14:58:38+00:00,37.0,NaN,74.0,<NA>,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine,2023-05-17 15:58:01,2023-05-17 15:58:38
4,4MLe,FOR11905,HeartRate,2023-05-17 18:00:55+00:00,NaT,NaN,NaN,95.0,<NA>,2023-05-17 00:00:00+00:00,18.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine,2023-05-17 19:00:55,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127777465,GE6w,FOR12092,CoveredDistance,2025-08-13 18:58:52+00:00,2025-08-13 18:58:55+00:00,3.0,29.717,NaN,<NA>,2025-08-13 00:00:00+00:00,18.0,2025-07-29 00:00:00+00:00,Lang,2025-08-16 07:44:11.112000+00:00,120.0,gps_single,2025-08-13 20:58:52,2025-08-13 20:58:55
127777466,GE6w,FOR12092,Steps,2025-08-13 18:58:52+00:00,2025-08-13 18:58:55+00:00,3.0,16.000,NaN,<NA>,2025-08-13 00:00:00+00:00,18.0,2025-07-29 00:00:00+00:00,Lang,2025-08-16 07:44:11.112000+00:00,120.0,gps_single,2025-08-13 20:58:52,2025-08-13 20:58:55
127777467,GE6w,FOR12092,ActiveBurnedCalories,2025-08-13 20:41:16+00:00,2025-08-13 20:41:28+00:00,12.0,0.329,NaN,<NA>,2025-08-13 00:00:00+00:00,20.0,2025-07-29 00:00:00+00:00,Lang,2025-08-16 07:44:11.113000+00:00,120.0,gps_single,2025-08-13 22:41:16,2025-08-13 22:41:28
127777468,GE6w,FOR12092,CoveredDistance,2025-08-13 20:41:16+00:00,2025-08-13 20:41:28+00:00,12.0,11.645,NaN,<NA>,2025-08-13 00:00:00+00:00,20.0,2025-07-29 00:00:00+00:00,Lang,2025-08-16 07:44:11.114000+00:00,120.0,gps_single,2025-08-13 22:41:16,2025-08-13 22:41:28


In [83]:
df_pass_act_loc =df_pass_act_base[df_pass_act_base.type.isin(["Latitude", "Longitude"])][["customer", "startTimestamp", "type", "doubleValue"]]

In [84]:

lat = (
    df_pass_act_loc.loc[df_pass_act_loc["type"].eq("Latitude"), ["customer","startTimestamp","doubleValue"]]
      .sort_values(["customer","startTimestamp"])
      .drop_duplicates(["customer","startTimestamp"], keep="first")
      .rename(columns={"doubleValue":"Latitude"})
)

lon = (
    df_pass_act_loc.loc[df_pass_act_loc["type"].eq("Longitude"), ["customer","startTimestamp","doubleValue"]]
      .sort_values(["customer","startTimestamp"])
      .drop_duplicates(["customer","startTimestamp"], keep="first")
      .rename(columns={"doubleValue":"Longitude"})
)

df_loc = lat.merge(lon, on=["customer","startTimestamp"], how="outer", copy=False)



In [85]:
df_loc.customer.nunique()

267

In [86]:
# Example usage with HDBSCAN and normalized min_samples:
extractor = gps_features.HomeClusterExtractor(df_loc, speed_limit=speed_limit, max_distance=max_distance, epsilon=epsilon, min_samples=min_samples, 
                                 min_nights_obs = min_nights_obs, min_f_home=min_f_home, clustering_method='dbscan', 
                                 normalize_min_samples=False, min_data_points=50)
result = extractor.run()


INFO:root:Data quality check: 248 customers with sufficient data retained.
INFO:root:Customers with no home after time-based method: 28
INFO:root:Fallback home clusters assigned: 28


In [87]:
# 20 customers not enough GPS data (i.e. less than 50 data points, so that no home cluster could be computed 


In [88]:
home_clusters_red = result[["customer", "startTimestamp", "at_home","transition", "distance", "stationary","time_diff", "speed", "clusterID", "homeID" ]]

In [89]:
gps_customer_list = home_clusters_red.customer.unique().tolist()

In [90]:
len(gps_customer_list)

248

## 2. Prepare EMA data

In [91]:
df_ema_udi = df_piv_filtered[["customer", "quest_create_day","quest_create", "unique_day_id", "assess",  "quest_create_hour", "weekday", 
                     "weekend","season", "time_of_day","n_quest","mean_na",
                     'situation1_active leisure','situation1_care work', 'situation1_chores',
       'situation1_eating - drinking - selfcare', 'situation1_other',
       'situation1_passive leisure', 'situation1_smartphone - social media',
       'situation1_travelling', 'situation1_work or study']]

In [92]:
df_ema_udi = df_ema_udi.loc[df_ema_udi.customer.isin(gps_customer_list)]

In [93]:
# Group by customer and unique_day_id and calculate the minimum quest_create
df_min_quest = df_ema_udi.groupby(['customer', 'unique_day_id'])['quest_create'].min().reset_index()

# Rename the column to sensor_block_end
df_min_quest.rename(columns={'quest_create': 'sensor_block_end'}, inplace=True)

# Merge the minimum quest_create back to the original DataFrame
df_ema_udi = pd.merge(df_ema_udi, df_min_quest, on=['customer', 'unique_day_id'], how='left')

# Create the sensor_block_start column, which is 2 hours before quest_create
df_ema_udi.drop(columns=['quest_create'], inplace=True)
df_ema_udi = df_ema_udi.drop_duplicates()

In [94]:
# prepare mapping of passing data by creating blocks

df_ema_udi['sensor_block_start'] = df_ema_udi['sensor_block_end'] - pd.Timedelta(hours=timedelta_hours)


In [95]:
# Only include first assessment phase

df_ema_udi_base = df_ema_udi.loc[df_ema_udi.assess == 0]

In [96]:
df_ema_udi_base = df_ema_udi_base.copy()
df_ema_udi_base["unique_blocks"] = df_ema_udi_base.customer + df_ema_udi_base.unique_day_id
df_ema_udi_base = df_ema_udi_base.drop_duplicates(subset = ["customer", "unique_blocks"])

In [97]:
df_ema_udi_merged = pd.merge(df_ema_udi_base, df_redcap, on="customer", how="left")

In [98]:
df_ema_udi_merged = df_ema_udi_merged.drop_duplicates(subset = ["customer", "unique_blocks"])

## 3. Merge EMA to passive data

In [99]:
# Ensure 'customer' columns are strings and stripped of whitespace
df_ema_udi_merged['customer'] = df_ema_udi_merged['customer'].astype(str).str.strip()
df_pass_act_base['customer'] = df_pass_act_base['customer'].astype(str).str.strip()
home_clusters_red['customer'] = home_clusters_red['customer'].astype(str).str.strip()

In [100]:
df_ema_udi_merged_mapper = df_ema_udi_merged[['customer','sensor_block_end', 'sensor_block_start','unique_blocks']]

In [101]:
df_pass_act_base_mapper = df_pass_act_base[['customer', 'type', 'startTimestamp', 'endTimestamp', 'doubleValue',
       'longValue', 'booleanValue']]

In [102]:
# run all EMA mappings
df_ema_enriched = run_ema_mappings(df_ema_udi_merged_mapper, df_pass_act_base_mapper, home_clusters_red)

Heart Rate Data Cleaning Summary:
Initial entries: 920617
Removed due to non-numeric values: 0
Removed due to thresholds: 0
Total entries removed: 0
Remaining entries: 920617
Processing user batch 1/13...
Processing user batch 2/13...
Processing user batch 3/13...
Processing user batch 4/13...
Processing user batch 5/13...
Processing user batch 6/13...
Processing user batch 7/13...
Processing user batch 8/13...
Processing user batch 9/13...
Processing user batch 10/13...
Processing user batch 11/13...
Processing user batch 12/13...
Processing user batch 13/13...
Step Data Cleaning Summary:
Initial entries: 879726
Removed due to non-numeric values: 0
Removed due to negative steps: 0
Removed due to zero or negative duration: 0
Removed due to exceeding steps per minute threshold: 689
Remaining entries: 879037
Processing user batch 1/13...
Processing user batch 2/13...
Processing user batch 3/13...
Processing user batch 4/13...
Processing user batch 5/13...
Processing user batch 6/13...
Pr

### Include weather data

In [103]:
import openmeteo_requests

import requests_cache
import pandas as pd
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 52.5244,
	"longitude": 13.4105,
	"start_date": "2023-05-01",
	"end_date": "2024-12-31",
	"daily": ["temperature_2m_max", "temperature_2m_min", "temperature_2m_mean", "apparent_temperature_max", "apparent_temperature_min", "apparent_temperature_mean", "sunshine_duration", "precipitation_sum", "precipitation_hours"],
	"timezone": "Europe/Berlin"
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop fo*+r multiple locations or weather models
response = responses[0]
print(f"Coordinates {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation {response.Elevation()} m asl")
print(f"Timezone {response.Timezone()} {response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0 {response.UtcOffsetSeconds()} s")

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()
daily_temperature_2m_mean = daily.Variables(2).ValuesAsNumpy()
daily_apparent_temperature_max = daily.Variables(3).ValuesAsNumpy()
daily_apparent_temperature_min = daily.Variables(4).ValuesAsNumpy()
daily_apparent_temperature_mean = daily.Variables(5).ValuesAsNumpy()
daily_sunshine_duration = daily.Variables(6).ValuesAsNumpy()
daily_precipitation_sum = daily.Variables(7).ValuesAsNumpy()
daily_precipitation_hours = daily.Variables(8).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
	end = pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["apparent_temperature_mean"] = daily_apparent_temperature_mean
daily_data["sunshine_duration"] = daily_sunshine_duration/3600
daily_data["precipitation_hours"] = daily_precipitation_hours

daily_dataframe = pd.DataFrame(data = daily_data)

Coordinates 52.5483283996582°N 13.407821655273438°E
Elevation 46.0 m asl
Timezone b'Europe/Berlin' b'GMT+1'
Timezone difference to GMT+0 3600 s


In [104]:
df_ema_udi_merged.columns.tolist()

['customer',
 'quest_create_day',
 'unique_day_id',
 'assess',
 'quest_create_hour',
 'weekday',
 'weekend',
 'season',
 'time_of_day',
 'n_quest',
 'mean_na',
 'situation1_active leisure',
 'situation1_care work',
 'situation1_chores',
 'situation1_eating - drinking - selfcare',
 'situation1_other',
 'situation1_passive leisure',
 'situation1_smartphone - social media',
 'situation1_travelling',
 'situation1_work or study',
 'sensor_block_end',
 'sensor_block_start',
 'unique_blocks',
 'for_id',
 'redcap_event_name',
 'basic_documentation_sheet_timestamp',
 'age',
 'gender',
 'scid_cv_prim_cat',
 'marital_status',
 'partnership',
 'graduation',
 'profession',
 'ema_start_date',
 'years_of_education',
 'employability',
 'ses',
 'ema_smartphone',
 'ema_sleep',
 'ema_watch',
 'prior_treatment',
 'ema_special_event',
 'psychotropic',
 'somatic_problems',
 'gender_description',
 'scid_cv_description',
 'marital_status_description',
 'employability_description',
 'employability_description_

In [105]:
df_ema_udi_merged_simple = df_ema_udi_merged[['customer', 'quest_create_day','unique_day_id', 'assess',
       'quest_create_hour', 'weekday', 'weekend', 'season', 'time_of_day',
       'n_quest', 'mean_na',  'unique_blocks', 'for_id', 'ema_watch','age', 'gender',
       'scid_cv_prim_cat', 'marital_status', 'partnership', 'graduation',
       'profession', 'ema_start_date', 'years_of_education', 'employability',
       'ses', 'ema_smartphone', 'ema_sleep', 'prior_treatment',
       'ema_special_event', 'psychotropic', 'somatic_problems',
       'gender_description', 'scid_cv_description',
       'marital_status_description', 'employability_description',
       'employability_description_simple',
       'prior_treatment_description_simple', 'graduation_description',
       'profession_description', 'prior_treatment_description',
       'ema_smartphone', 'ema_special_event_description',
       'age_description', 'somatic_description', 'psychotropic_description', 
       'situation1_active leisure','situation1_care work','situation1_chores','situation1_eating - drinking - selfcare',
       'situation1_other','situation1_passive leisure','situation1_smartphone - social media','situation1_travelling',
       'situation1_work or study',]]

In [106]:
df_ema_enriched = df_ema_enriched.merge(df_ema_udi_merged_simple, on= ["customer", "unique_blocks"])

In [107]:
df_ema_enriched.customer.nunique()

248

In [108]:
daily_dataframe['date'] = pd.to_datetime(daily_dataframe['date'], utc=True)
daily_dataframe['assessment_day'] = daily_dataframe['date'].dt.date
df_ema_enriched["quest_create_day"] = pd.to_datetime(df_ema_enriched["quest_create_day"]).dt.date

df_ema_weather = pd.merge(
    df_ema_enriched,
    daily_dataframe,
    left_on="quest_create_day",
    right_on='assessment_day',
    how='left'  # Use 'left' to keep all records from df_ema_enriched
)

# Drop redundant columns if necessary
df_ema_weather.drop(['date', 'assessment_day'], axis=1, inplace=True)

### Analyze missingness

In [109]:
feature_group_pa = [
    'activity_102_minutes',
       'activity_103_minutes', 'activity_104_minutes', 'activity_105_minutes',
       'activity_106_minutes', 'activity_107_minutes'
]
feature_group_hr = ['hr_mean', 'hr_min', 'hr_max', 'hr_std', 'hr_zone_resting', 'hr_zone_moderate',
       'hr_zone_vigorous']

feature_group_gps = [ 'n_GPS', 'total_distance_km', 'at_home_minute',
       'time_in_transition_minutes', 'time_stationary_minutes']

feature_group_weather = ["apparent_temperature_mean", "sunshine_duration", "precipitation_hours"]

feature_group_person_static = [
    'age',
    'somatic_description',
    'psychotropic_description',
    'employability_description_simple',
    'prior_treatment_description_simple',
    'ema_smartphone_description'
]

columns_to_check = ['activity_102_minutes',
       'activity_103_minutes', 'activity_104_minutes', 'activity_105_minutes',
       'activity_106_minutes', 'activity_107_minutes', 'hr_mean', 'hr_min', 'hr_max', 'hr_std', 'hr_median', 'range_heartrate',
       'iqr_heartrate', 'skewness_heartrate', 'kurtosis_heartrate','hr_peak_counts', 'hr_zone_resting', 'hr_zone_moderate','hr_zone_vigorous', 
                    'n_GPS', 'total_distance_km', 'at_home_minute','time_in_transition_minutes', 'time_stationary_minutes', "apparent_temperature_mean", 
                    "sunshine_duration", "precipitation_hours",'n_steps', 'calories_burned', 'age','somatic_description','psychotropic_description',
                    'employability_description_simple', 'prior_treatment_description_simple', 'ema_smartphone_description']

#### Clean Rows where All Features are missing

In [110]:
# Define the columns to check for missing values
cols_to_check = [
    'activity_102_minutes', 'activity_103_minutes', 'activity_104_minutes',
    'activity_105_minutes', 'activity_106_minutes', 'activity_107_minutes',
    'hr_mean', 'hr_min', 'hr_max', 'hr_std', 'hr_zone_resting', 'hr_zone_moderate', 'hr_zone_vigorous',
    'n_GPS', 'total_distance_km', 'at_home_minute',
    'time_in_transition_minutes', 'time_stationary_minutes'
]

# Check rows where all the specified columns have the value -1
all_negative = (df_ema_weather[cols_to_check] == -1).all(axis=1)

# Calculate the number of rows to be removed
num_removed = all_negative.sum()

# Remove rows where all the specified columns are -1
df_ema_weather_cleaned = df_ema_weather[~all_negative].reset_index(drop=True)

# Print the shape of the cleaned DataFrame and number of removed entries
print("Original DataFrame shape:", df_ema_weather.shape)
print("Cleaned DataFrame shape:", df_ema_weather_cleaned.shape)
print("Number of removed entries:", num_removed)


Original DataFrame shape: (20303, 78)
Cleaned DataFrame shape: (19999, 78)
Number of removed entries: 304


#### Clean Rows where more than 50% of GPS data are missing

In [111]:
df_ema_weather_cleaned

,customer,sensor_block_end,sensor_block_start,unique_blocks,hr_mean,hr_min,hr_max,hr_std,hr_zone_resting,hr_zone_moderate,...,situation1_chores,situation1_eating - drinking - selfcare,situation1_other,situation1_passive leisure,situation1_smartphone - social media,situation1_travelling,situation1_work or study,apparent_temperature_mean,sunshine_duration,precipitation_hours
0,05kz,2023-10-10 10:07:04.089000+00:00,2023-10-10 08:07:04.089000+00:00,05kz20231010_2.0,62.333333,46.0,80.0,13.912425,1.133333,1.266667,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,16.338188,9.358354,0.0
1,05kz,2023-10-10 11:51:32.262000+00:00,2023-10-10 09:51:32.262000+00:00,05kz20231010_3.0,64.400000,59.0,71.0,4.715930,0.833333,3.266667,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,16.338188,9.358354,0.0
2,05kz,2023-10-10 15:35:25.929000+00:00,2023-10-10 13:35:25.929000+00:00,05kz20231010_5.0,70.000000,64.0,81.0,6.244998,0.000000,5.816667,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,16.338188,9.358354,0.0
3,05kz,2023-10-10 17:43:45.973000+00:00,2023-10-10 15:43:45.973000+00:00,05kz20231010_6.0,63.333333,55.0,81.0,7.717225,3.766667,5.183333,...,1.0,1.0,0.0,1.0,1.0,0.0,0.0,16.338188,9.358354,0.0
4,05kz,2023-10-10 19:30:34.913000+00:00,2023-10-10 17:30:34.913000+00:00,05kz20231010_7.0,72.250000,66.0,79.0,3.897114,0.000000,8.750000,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,16.338188,9.358354,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19994,zgxc,2023-11-09 18:10:05.993000+00:00,2023-11-09 16:10:05.993000+00:00,zgxc20231109_6.0,66.666667,48.0,72.0,6.368324,1.466667,8.116667,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,4.699565,6.810548,2.0
19995,zgxc,2023-11-10 11:04:41.090000+00:00,2023-11-10 09:04:41.090000+00:00,zgxc20231110_2.0,69.333333,56.0,89.0,10.561986,1.966667,6.633333,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.748881,6.959469,3.0
19996,zgxc,2023-11-10 16:37:37.646000+00:00,2023-11-10 14:37:37.646000+00:00,zgxc20231110_5.0,83.555556,67.0,161.0,27.897077,0.000000,6.966667,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2.748881,6.959469,3.0
19997,zgxc,2023-11-10 22:27:49.683000+00:00,2023-11-10 20:27:49.683000+00:00,zgxc20231110_8.0,62.000000,61.0,63.0,1.000000,0.000000,1.250000,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2.748881,6.959469,3.0


In [112]:
# Calculate total beeps and condition beeps per participant
beep_counts = df_ema_weather_cleaned.groupby('customer').agg(
    total_beeps=('unique_blocks', 'count'),
    condition_beeps=('at_home_minute', lambda x: ((x == -1) & (df_ema_weather_cleaned.loc[x.index, 'n_steps'] > 625)).sum())
).reset_index()

# Calculate the ratio of condition beeps to total beeps
beep_counts['condition_beeps_ratio'] = beep_counts['condition_beeps'] / beep_counts['total_beeps']

beep_counts['condition_beeps'].sum()

1871

In [113]:
beep_counts = df_ema_weather_cleaned.groupby('customer').agg(
    total_beeps=('unique_blocks', 'count'),
    condition_beeps=(
        'at_home_minute',
        lambda x: (
            (x == -1)
            & (df_ema_weather_cleaned.loc[x.index, 'n_steps'] > 625)
            & (df_ema_weather_cleaned.loc[x.index, 'situation1_travelling'] == 1)
        ).sum()
    )
).reset_index()

beep_counts['condition_beeps_ratio'] = (
    beep_counts['condition_beeps'] / beep_counts['total_beeps']
)

beep_counts['condition_beeps'].sum()

726

In [114]:
# Identify participants where condition beeps >50% of total beeps
high_condition_participants = beep_counts[beep_counts['condition_beeps_ratio'] > 0.50]['customer']

# Remove those participants 
df_ema_enriched_filtered = df_ema_weather_cleaned[~df_ema_weather_cleaned['customer'].isin(high_condition_participants)].copy()


In [115]:
df_ema_enriched_filtered.customer.nunique()

248

In [116]:
group_missing_df = summarize_missing_data(
    df=df_ema_enriched_filtered,
    feature_group_pa=feature_group_pa,
    feature_group_gps=feature_group_gps,
    feature_group_hr=feature_group_hr,
    feature_group_weather = feature_group_weather,
    feature_group_person_static = feature_group_person_static,
    columns_to_check=columns_to_check,
    customer_id_col = "customer")


=== Missing Data Analysis ===

---- Person-Static Features Missingness (Based on Unique Customers) ----

  age: 0 missing [ 0.00% of unique customers ]
  somatic_description: 0 missing [ 0.00% of unique customers ]
  psychotropic_description: 0 missing [ 0.00% of unique customers ]
  employability_description_simple: 0 missing [ 0.00% of unique customers ]
  prior_treatment_description_simple: 0 missing [ 0.00% of unique customers ]

---- Group-wise Missing Data Summary ----

Group: PA (contains 6 column(s))
--------------------------------------------------
  activity_102_minutes: 11732 missing (-1)  [58.66% of total rows]
  activity_103_minutes: 11732 missing (-1)  [58.66% of total rows]
  activity_104_minutes: 11732 missing (-1)  [58.66% of total rows]
  activity_105_minutes: 11732 missing (-1)  [58.66% of total rows]
  activity_106_minutes: 11732 missing (-1)  [58.66% of total rows]
  activity_107_minutes: 11732 missing (-1)  [58.66% of total rows]

Group: GPS (contains 5 column(s)

### GPS condition

In [117]:

conditions = [
    (df_ema_enriched_filtered['n_GPS'] == -1) & (df_ema_enriched_filtered['n_steps'] > 625),
    (df_ema_enriched_filtered['n_GPS'] == -1) & (df_ema_enriched_filtered['n_steps'] <= 625),
    (df_ema_enriched_filtered['n_GPS'] != -1)
]
# Define the corresponding choices
choices = [
    'Steps>625',
    'Steps<=625',
    'GPS_present'
]

# Create the categorical column
df_ema_enriched_filtered['missing_GPS'] = np.select(conditions, choices, default='Unknown')


In [118]:
df_ema_enriched_filtered.groupby('missing_GPS')['customer'].count()

missing_GPS
GPS_present    14973
Steps<=625      3155
Steps>625       1871
Name: customer, dtype: int64

### Steps condition

In [119]:
# Define the condition for all specified columns being -1
cols_to_check = [
    'hr_mean', 
    'activity_102_minutes', 
    'activity_103_minutes', 
    'activity_104_minutes', 
    'activity_105_minutes', 
    'activity_106_minutes', 
    'activity_107_minutes'
]

all_negative = (df_ema_enriched_filtered[cols_to_check] == -1).all(axis=1)

# Define the conditions for the new "missing_steps" column
conditions_steps = [
    # Condition: n_steps is -1 and all other specified columns are -1
    (df_ema_enriched_filtered['n_steps'] == -1) & all_negative,
    # Condition: n_steps is -1 but at least one of the specified columns is not -1
    (df_ema_enriched_filtered['n_steps'] == -1) & (~all_negative),
    # Condition: n_steps is not -1
    (df_ema_enriched_filtered['n_steps'] != -1)
]

# Define the corresponding choices
choices_steps = ['step_missing', 'step_zero', 'not_missing']

# Create the "missing_steps" column
df_ema_enriched_filtered['missing_steps'] = np.select(conditions_steps, choices_steps, default='Unknown')


In [120]:
df_ema_enriched_filtered.groupby('missing_steps')['customer'].count()

missing_steps
not_missing     18647
step_missing      521
step_zero         831
Name: customer, dtype: int64

### PA condition

In [121]:
# Define the conditions for the new "missing_pa" column

conditions_pa = [
    # 1. If 'activity_102_minutes' is NOT -1, mark as "not_missing"
    (df_ema_enriched_filtered['activity_102_minutes'] != -1),
    
    # 2. If 'activity_102_minutes' is -1 and both 'n_steps' and 'hr_mean' are -1, mark as "pa_missing"
    (df_ema_enriched_filtered['activity_102_minutes'] == -1) &
    ((df_ema_enriched_filtered['n_steps'] == -1) & (df_ema_enriched_filtered['hr_mean'] == -1)),
    
    # 3. If 'activity_102_minutes' is -1 and at least one of 'n_steps' or 'hr_mean' is not -1, mark as "pa_zero"
    (df_ema_enriched_filtered['activity_102_minutes'] == -1) &
    (((df_ema_enriched_filtered['n_steps'] != -1) | (df_ema_enriched_filtered['hr_mean'] != -1)))
]

# Define the corresponding choices for each condition
choices_pa = ['not_missing', 'pa_missing', 'pa_zero']

# Create the "missing_pa" column using np.select
df_ema_enriched_filtered['missing_pa'] = np.select(conditions_pa, choices_pa, default='Unknown')


In [122]:
df_ema_enriched_filtered.groupby('missing_pa')['customer'].count()

missing_pa
not_missing     8267
pa_missing       521
pa_zero        11211
Name: customer, dtype: int64

### HR condition

In [123]:
# Define the condition for all specified columns being -1
cols_to_check = [
    'n_steps', 
    'activity_102_minutes', 
    'activity_103_minutes', 
    'activity_104_minutes', 
    'activity_105_minutes', 
    'activity_106_minutes', 
    'activity_107_minutes'
]

all_negative = (df_ema_enriched_filtered[cols_to_check] == -1).all(axis=1)

# Define the conditions for the new "missing_steps" column
conditions_steps = [
    # Condition: hr is -1 and all other specified columns are -1
    (df_ema_enriched_filtered['hr_mean'] == -1) & all_negative,
    # Condition: hr is -1 but at least one of the specified columns is not -1
    (df_ema_enriched_filtered['hr_mean'] == -1) & (~all_negative),
    # Condition: hr is not -1
    (df_ema_enriched_filtered['hr_mean'] != -1)
]

# Define the corresponding choices
choices_steps = ['hr_missing', 'hr_missing', 'not_missing']

# Create the "missing_steps" column
df_ema_enriched_filtered['missing_hr'] = np.select(conditions_steps, choices_steps, default='Unknown')


In [124]:
df_ema_enriched_filtered.groupby('missing_hr')['customer'].count()

missing_hr
hr_missing       780
not_missing    19219
Name: customer, dtype: int64

In [125]:
# Calculate the number of unique 'unique_day_id' per 'customer' and 'quest_complete_day'
df_ema_enriched_filtered['n_quest'] = df_ema_enriched_filtered.groupby(['customer', 'quest_create_day'])['unique_day_id'].transform('nunique')

# 1) For each customer, count how many UNIQUE days they have n_quest >= min_num_daily
df_days_count = (df_ema_enriched_filtered[df_ema_enriched_filtered['n_quest'] >= min_num_daily]
                 .groupby('customer')['quest_create_day'].nunique()
                 .reset_index(name='n_days_min'))

# 2) Identify valid customers
valid_customers = df_days_count.loc[df_days_count['n_days_min'] >= min_days_data, 'customer']

# 3) Filter the original df_piv to keep all rows from valid customers
df_ema_enriched_filtered = df_ema_enriched_filtered[df_ema_enriched_filtered['customer'].isin(valid_customers)]

In [126]:
group_missing_df = summarize_missing_data(
    df=df_ema_enriched_filtered,
    feature_group_pa=feature_group_pa,
    feature_group_gps=feature_group_gps,
    feature_group_hr=feature_group_hr,
    feature_group_weather = feature_group_weather,
    feature_group_person_static = feature_group_person_static,
    columns_to_check=columns_to_check,
    customer_id_col = "customer")

=== Missing Data Analysis ===

---- Person-Static Features Missingness (Based on Unique Customers) ----

  age: 0 missing [ 0.00% of unique customers ]
  somatic_description: 0 missing [ 0.00% of unique customers ]
  psychotropic_description: 0 missing [ 0.00% of unique customers ]
  employability_description_simple: 0 missing [ 0.00% of unique customers ]
  prior_treatment_description_simple: 0 missing [ 0.00% of unique customers ]

---- Group-wise Missing Data Summary ----

Group: PA (contains 6 column(s))
--------------------------------------------------
  activity_102_minutes: 11571 missing (-1)  [58.46% of total rows]
  activity_103_minutes: 11571 missing (-1)  [58.46% of total rows]
  activity_104_minutes: 11571 missing (-1)  [58.46% of total rows]
  activity_105_minutes: 11571 missing (-1)  [58.46% of total rows]
  activity_106_minutes: 11571 missing (-1)  [58.46% of total rows]
  activity_107_minutes: 11571 missing (-1)  [58.46% of total rows]

Group: GPS (contains 5 column(s)

In [127]:
df_ema_enriched_filtered.customer.nunique()

242

In [128]:
with open(preprocessed_path_jitai +'/map_ema_passive.pkl', 'wb') as file:
    pickle.dump(df_ema_enriched_filtered, file)

In [129]:
df_ema_enriched_filtered.mean_na.std()

1.0807947315763091